# GPT-4 Zero-Shot — Medical Abstracts

Reproducible zero-shot evaluation from *Zero-Shot vs. Fine-Tuned* (structured prompts, constraint-based parsing, temperature=0). Requires `OPENAI_API_KEY`.

Legacy T5 zero-shot notebooks are in `archieve/`.


In [7]:
import os
import sys

PROJECT_ROOT = os.getcwd()
if not os.path.isdir(os.path.join(PROJECT_ROOT, 'zeroshot')):
    PROJECT_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.environ.setdefault('WANDB_DISABLED', 'true')


'true'

In [ ]:
from zeroshot.classifier import ZeroShotClassifier
from zeroshot.evaluate import evaluate_zero_shot, load_split_df, resolve_base_dir
from zeroshot.labels import DATASET_CONFIGS
from dotenv import load_dotenv

load_dotenv()  # automatically finds .env in parent dirs

CONFIG_KEY = 'medical_abstract'
BASE_DIR = resolve_base_dir()
MAX_SAMPLES = int(os.getenv('ZEROSHOT_MAX_SAMPLES', '0')) or None  # cap for dry runs

# gpt-4o-mini is cheaper for development; use gpt-4 for paper replication
MODEL = os.getenv('ZEROSHOT_MODEL', 'gpt-4o-mini')
BACKEND = os.getenv('ZEROSHOT_BACKEND', 'openai')  # or mistral_local

classifier = ZeroShotClassifier(model=MODEL, backend=BACKEND, temperature=0.0, max_tokens=10)
config = DATASET_CONFIGS[CONFIG_KEY]
print(f'Dataset: {config.name} | Model: {MODEL} | Base: {BASE_DIR}')

In [9]:
val_df = load_split_df(config, 'validation', BASE_DIR)
test_df = load_split_df(config, 'test', BASE_DIR)
print(f'Validation: {len(val_df)} | Test: {len(test_df)}')
print(val_df[config.label_column].value_counts().head())


Validation: 6611 | Test: 2770
label
4    2119
0    1543
3    1445
2     848
1     656
Name: count, dtype: int64


In [10]:
val_results = evaluate_zero_shot(
    val_df,
    CONFIG_KEY,
    classifier,
    dataset_name='validation',
    max_samples=MAX_SAMPLES,
    results_dir=os.path.join(PROJECT_ROOT, 'Results'),
)


Zero-shot inference: 100%|██████████| 6611/6611 [1:09:09<00:00,  1.59it/s]



ZERO-SHOT EVALUATION — MEDICAL ABSTRACTS (validation)
Model: gpt-4.1-mini | Backend: openai
Valid pairs: 6611/6611
Accuracy:     0.6699
F1 (macro):   0.6592
F1 (weighted): 0.6533
Precision:    0.6477
Recall:       0.6978

Classification report:
                      precision    recall  f1-score   support

           NEOPLASMS       0.79      0.87      0.83      1543
           DIGESTIVE       0.56      0.68      0.61       656
             NERVOUS       0.53      0.69      0.60       848
      CARDIOVASCULAR       0.70      0.88      0.78      1445
GENERAL_PATHOLOGICAL       0.65      0.37      0.47      2119

            accuracy                           0.67      6611
           macro avg       0.65      0.70      0.66      6611
        weighted avg       0.67      0.67      0.65      6611


Saved confusion matrix: /Users/kirthi/Documents/UCBerkeley/kirthi_portfolio/MIDS/Academic_Projects/Medical_NLP_Zeroshot_vs_Finetune_v2/Results/zeroshot_confusion_matrix_medical_abstract_valida

In [11]:
test_results = evaluate_zero_shot(
    test_df,
    CONFIG_KEY,
    classifier,
    dataset_name='test',
    max_samples=MAX_SAMPLES,
    results_dir=os.path.join(PROJECT_ROOT, 'Results'),
)


Zero-shot inference: 100%|██████████| 2770/2770 [29:49<00:00,  1.55it/s]


ZERO-SHOT EVALUATION — MEDICAL ABSTRACTS (test)
Model: gpt-4.1-mini | Backend: openai
Valid pairs: 2770/2770
Accuracy:     0.6444
F1 (macro):   0.6440
F1 (weighted): 0.6190
Precision:    0.6317
Recall:       0.6939

Classification report:
                      precision    recall  f1-score   support

           NEOPLASMS       0.73      0.85      0.79       611
           DIGESTIVE       0.59      0.73      0.65       283
             NERVOUS       0.56      0.70      0.62       369
      CARDIOVASCULAR       0.65      0.88      0.74       587
GENERAL_PATHOLOGICAL       0.64      0.31      0.42       920

            accuracy                           0.64      2770
           macro avg       0.63      0.69      0.64      2770
        weighted avg       0.64      0.64      0.62      2770


Saved confusion matrix: /Users/kirthi/Documents/UCBerkeley/kirthi_portfolio/MIDS/Academic_Projects/Medical_NLP_Zeroshot_vs_Finetune_v2/Results/zeroshot_confusion_matrix_medical_abstract_test_openai.

## Optional: Mistral-7B-Instruct (local)

Set `ZEROSHOT_BACKEND=mistral_local` and run on a GPU. Uses 4-bit quantization by default.
